# Week 2: Exercise 5 - Complete Agent Loop

**Goal:** Combine tool calling + error handling into one robust agent.


In [23]:
import os
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    api_key=os.environ.get("GEMINI_API_KEY"),
)
print("API client ready")


API client ready


## Step 1: Define Tools
**TODO:** Define schemas and implementations for `get_time` and `add_numbers`.


In [24]:
import json, time

TOOLS = [
    # TODO: schema for get_time and add_numbers
    # Hint: two {"type": "function", "function": {...}} dicts — copy the shape
    #       from exercise 2, add "parameters" with properties for add_numbers
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "Return current time as 'YYYY-MM-DD HH:MM:SS'.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            }
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add a and b together",
            "parameters": {
                "type": "object",
                "properties": {
                    'a': {"type": "integer", "description": "Any number"},
                    'b': {"type": "integer", "description": "Any number"}
                },
                "required": ['a','b'],
            }
        },
    }
]

def get_time():
    # TODO: same as exercise 3
    return datetime.now().strftime("%Y-%M-%D %H:%M:%S")

def add_numbers(a: int, b: int):
    # TODO
    return a + b

def call_function(name, args):
    # TODO: same dispatch pattern as exercises 2-3
    func = {
        "get_time": get_time,
        "add_numbers": add_numbers
    }

    try:
        if name in func:
            return func[name](**args)
        else: raise ValueError(f"Function {name} not found!")
    except TypeError as TE:
        return f"Error: {TE}"

## Step 2: Implement classify_error
**TODO:** Classify errors as retryable or permanent.


In [25]:
def classify_error(error):
    """TODO: Return (error_type, is_retryable)."""
    # Hint: same as exercise 4's ErrorClassifier, as a plain function
    retryable = ["rate limit", "timeout", "502", "503", "504", "429", "overloaded"]
    for err_type in retryable:
        if err_type.lower() in str(error).lower():
            return ("retryable", True)
    permanent = ["401", "403", "invalid api key", "invalid model"]
    for err_type in permanent:
        if err_type.lower() in str(error).lower():
            return ("permanent", False)
        
    return ("unknown", False)


## Step 3: Implement run_conversation

**TODO:** Full agent loop with error handling + tool calling.

**IMPORTANT:** Use `model_dump(exclude_none=True)` — Gemini rejects None values.


In [26]:
def run_conversation(user_input, max_iterations=10):
    """TODO: Full agent loop with error handling + tool calling."""
    # Hint: combine exercise 3's loop with exercise 4's retries:
    #       outer loop = agent iterations, inner loop = retry attempts
    #       on API error: classify; retryable -> sleep(2**attempt); else return None
    #       on tool_calls: execute + append tool messages (exercise 3 pattern)
    #       else: return result.content
    messages = [{"role": "system", "content": "You're an agent with tools!"},
                {"role": "user", "content": user_input}]
    for iter in range(max_iterations):
        for attempt in range(9):
            try:
                response = client.chat.completions.create(
                    model = os.environ.get("GEMINI_3.6_MODEL"),
                    messages = messages,
                    tools = TOOLS
                )

                message = response.choices[0].message
                messages.append(message.model_dump(exclude_none = True))

                if not message.tool_calls:
                    return message.content

                for tool in message.tool_calls:
                    func_name = tool.function.name
                    func_args = json.loads(tool.function.arguments)

                    func_response = call_function(func_name,func_args)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool.id,
                        "name": func_name,
                        "content": json.dumps(func_response)
                    })
                    
                break
            except Exception as E:
                err_type, retryable = classify_error(E)

                if not retryable or attempt == 8:
                    print(f"Giving up on errror {E}")
                    return None

                time.sleep(2**attempt)


## Step 4: Test It


In [27]:
# Test 1: classify_error
t, r = classify_error(Exception("429 Rate limit exceeded"))
assert r == True
print(f"  '429' -> {t}, retryable={r}")

t, r = classify_error(Exception("401 Unauthorized"))
assert r == False
print(f"  '401' -> {t}, retryable={r}")
print("Test 1 passed")


  '429' -> retryable, retryable=True
  '401' -> permanent, retryable=False
Test 1 passed


In [28]:
# Test 2: call_function
result = call_function("get_time", {})
assert isinstance(result, str)
print(f"  get_time() -> {result}")

result = call_function("add_numbers", {"a": 5, "b": 3})
assert result == 8
print(f"  add_numbers(5, 3) -> {result}")
print("Test 2 passed")


  get_time() -> 2026-17-08/25/26 12:17:13
  add_numbers(5, 3) -> 8
Test 2 passed


In [29]:
# Test 3: Agent loop with tools
result = run_conversation("What is 10 + 20?")
assert result is not None
assert "30" in result
print(f"  A: {result}")
print("Test 3 passed")


  A: 10 + 20 = 30
Test 3 passed


In [30]:
# Test 4: Multiple tools
result = run_conversation("What time is it and what is 7 + 8?")
assert result is not None
assert "15" in result
print(f"  A: {result}")
print("Test 4 passed")


  A: The current time is **2026-17-08/25/26 12:17:54** and **7 + 8 = 15**.
Test 4 passed


## Key Takeaways
- A production agent wraps every API call in retry logic
- Tool execution errors are caught and reported to the LLM
